In [ ]:
import os
import glob
import random
import torch
import itertools
import datetime
import time
import sys
import argparse
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.autograd import Variable
from PIL import Image
from torchvision.utils import save_image, make_grid
import lpips


#########################################################################
############################  datasets.py  ###################################

## If the input dataset is grayscale images, convert the images to RGB (not needed for the facades dataset used here)
def to_rgb(image):
    rgb_image = Image.new("RGB", image.size)
    rgb_image.paste(image)
    return rgb_image


## Build the dataset
class ImageDataset(Dataset):
    def __init__(self, root, transforms_=None, unaligned=False,
                 mode="train"):  ## (root = "./datasets/facades", unaligned=True: unaligned data)
        self.transform = transforms.Compose(transforms_)  ## Transform to tensor data
        self.unaligned = unaligned

        self.files_A = sorted(glob.glob(os.path.join(root, "%sA" % mode) + "/*.*"))  ## "./datasets/facades/trainA/*.*"
        self.files_B = sorted(glob.glob(os.path.join(root, "%sB" % mode) + "/*.*"))  ## "./datasets/facades/trainB/*.*"

    def __getitem__(self, index):
        image_A = Image.open(self.files_A[index % len(self.files_A)])  ## Take one image from A

        if self.unaligned:  ## If using unaligned data, randomly take one image from B
            image_B = Image.open(self.files_B[random.randint(0, len(self.files_B) - 1)])
        else:
            image_B = Image.open(self.files_B[index % len(self.files_B)])

        # If the image is grayscale, convert it to RGB
        if image_A.mode != "RGB":
            image_A = to_rgb(image_A)
        if image_B.mode != "RGB":
            image_B = to_rgb(image_B)

        # Convert RGB images to tensor for easier computation, return dictionary data
        item_A = self.transform(image_A)
        item_B = self.transform(image_B)
        return {"A": item_A, "B": item_B}

    ## Get the length of data A and B
    def __len__(self):
        return max(len(self.files_A), len(self.files_B))

In [ ]:

#########################################################################
############################  utils.py  ###################################

# Buffer for previously generated samples
class ReplayBuffer:
    def __init__(self, max_size=50):
        assert max_size > 0, "Empty buffer or trying to create a black hole. Be careful."
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, data):  # Push an image into the buffer and then pop one out
        to_return = []  # Ensure randomness of data to improve discriminator's ability to distinguish real and fake images
        for element in data.data:
            element = torch.unsqueeze(element, 0)
            if len(self.data) < self.max_size:  # If the buffer is not full, keep adding images
                self.data.append(element)
                to_return.append(element)
            else:
                if random.uniform(0, 1) > 0.5:  # If the buffer is full, with 50% probability, take an image from the buffer or use the current input image
                    i = random.randint(0, self.max_size - 1)
                    to_return.append(self.data[i].clone())
                    self.data[i] = element
                else:
                    to_return.append(element)
        return Variable(torch.cat(to_return))


# Set the learning rate as the initial learning rate multiplied by the value of the given lr_lambda function
class LambdaLR:
    def __init__(self, n_epochs, offset, decay_start_epoch):  # (n_epochs = 50, offset = epoch, decay_start_epoch = 30)
        assert (
                           n_epochs - decay_start_epoch) > 0, "Decay must start before the training session ends!"  # Assertion: Ensure n_epochs > decay_start_epoch
        self.n_epochs = n_epochs
        self.offset = offset
        self.decay_start_epoch = decay_start_epoch

    def step(self, epoch):  # return 1 - max(0, epoch - 30) / (50 - 30)
        return 1.0 - max(0, epoch + self.offset - self.decay_start_epoch) / (self.n_epochs - self.decay_start_epoch)

In [ ]:


#########################################################################
############################  models.py  ###################################

## Define the parameter initialization function
def weights_init_normal(m):
    classname = m.__class__.__name__  ## m is a formal parameter that can theoretically pass many things. To enable multiple arguments, each module must provide its own name. This line returns the name of m.
    if classname.find("Conv") != -1:  ## find(): Checks if "Conv" is in classname. Returns -1 if not found; otherwise, returns 0.
        torch.nn.init.normal_(m.weight.data, 0.0,
                              0.02)  ## m.weight.data represents the weights to be initialized. nn.init.normal_(): Random initialization using a normal distribution with mean 0 and standard deviation 0.02.
        if hasattr(m, "bias") and m.bias is not None:  ## hasattr(): Checks if m contains the attribute "bias" and if it is not empty.
            torch.nn.init.constant_(m.bias.data, 0.0)  ## nn.init.constant_(): Initializes the bias as a constant 0.
    elif classname.find("BatchNorm2d") != -1:  ## find(): Checks if "BatchNorm2d" is in classname. Returns -1 if not found; otherwise, returns 0.
        torch.nn.init.normal_(m.weight.data, 1.0,
                              0.02)  ## m.weight.data represents the weights to be initialized. nn.init.normal_(): Random initialization using a normal distribution with mean 1.0 and standard deviation 0.02.
        torch.nn.init.constant_(m.bias.data, 0.0)  ## nn.init.constant_(): Initializes the bias as a constant 0.

##############################
## Self-Attention Mechanism
##############################
class EfficientSelfAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):  # Added reduction_ratio
        super(EfficientSelfAttention, self).__init__()
        self.in_channels = in_channels
        self.reduction_ratio = reduction_ratio

        # Define query, key, and value convolutional layers
        self.query = nn.Conv2d(in_channels, in_channels // reduction_ratio, kernel_size=1)
        self.key = nn.Conv2d(in_channels, in_channels // reduction_ratio, kernel_size=1)
        self.value = nn.Conv2d(in_channels, in_channels, kernel_size=1)

        # Learnable scaling parameter
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        batch_size, C, height, width = x.size()

        # Downsample the feature map
        x_reduced = nn.functional.interpolate(x, scale_factor=1/self.reduction_ratio, mode='nearest')
        _, _, reduced_height, reduced_width = x_reduced.size()

        # Compute query, key, and value
        query = self.query(x_reduced).view(batch_size, -1, reduced_height * reduced_width).permute(0, 2, 1)  # (B, H'*W', C//r)
        key = self.key(x_reduced).view(batch_size, -1, reduced_height * reduced_width)  # (B, C//r, H'*W')
        value = self.value(x_reduced).view(batch_size, -1, reduced_height * reduced_width)  # (B, C, H'*W')

        # Compute attention scores
        attention = torch.bmm(query, key)  # (B, H'*W', H'*W')
        attention = torch.softmax(attention, dim=-1)  # Normalize along rows

        # Weighted sum
        out = torch.bmm(value, attention.permute(0, 2, 1))  # (B, C, H'*W')
        out = out.view(batch_size, C, reduced_height, reduced_width)

        # Upsample back to original size
        out = nn.functional.interpolate(out, size=(height, width), mode='nearest')

        # Add residual connection
        out = self.gamma * out + x  # Learnable scaling parameter gamma
        return out


##############################
## Residual Block
##############################
class ResidualBlock(nn.Module):
    def __init__(self, in_features):
        super(ResidualBlock, self).__init__()

        self.block = nn.Sequential(  ## block = [pad + conv + norm + relu + pad + conv + norm]
            nn.ReflectionPad2d(1),  ## ReflectionPad2d(): Pads the input tensor using the reflection of the input boundary
            nn.Conv2d(in_features, in_features, 3),  ## Convolution
            nn.InstanceNorm2d(in_features),  ## InstanceNorm2d(): Normalizes over HxW for each image, used in style transfer
            nn.ReLU(inplace=True),  ## Non-linear activation
            nn.ReflectionPad2d(1),  ## ReflectionPad2d(): Pads the input tensor using the reflection of the input boundary
            nn.Conv2d(in_features, in_features, 3),  ## Convolution
            nn.InstanceNorm2d(in_features),  ## InstanceNorm2d(): Normalizes over HxW for each image, used in style transfer
        )

    def forward(self, x):  ## Input is an image
        return x + self.block(x)  ## Output is the image plus the residual output of the network


##############################
## Generator Network (GeneratorResNet)
##############################
class Generator(nn.Module):
    def __init__(self, input_shape, num_residual_blocks):
        super(Generator, self).__init__()

        channels = input_shape[0]
        out_features = 64
        model = [
            nn.ReflectionPad2d(channels),
            nn.Conv2d(channels, out_features, 7),
            nn.InstanceNorm2d(out_features),
            nn.ReLU(inplace=True),
        ]
        in_features = out_features

        # Downsampling part
        for _ in range(2):
            out_features *= 2
            model += [
                nn.Conv2d(in_features, out_features, 3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True),
                EfficientSelfAttention(out_features),  # Introduce self-attention mechanism
            ]
            in_features = out_features

        # Residual blocks part
        for _ in range(num_residual_blocks):
            model += [ResidualBlock(out_features)]

        # Upsampling part
        for _ in range(2):
            out_features //= 2
            model += [
                nn.Upsample(scale_factor=2),
                nn.Conv2d(in_features, out_features, 3, stride=1, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True),
                EfficientSelfAttention(out_features),  # Introduce self-attention mechanism
            ]
            in_features = out_features

        # Output layer
        model += [
            nn.ReflectionPad2d(channels),
            nn.Conv2d(out_features, channels, 7),
            nn.Tanh(),
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

##############################
#        Discriminator
##############################
class Discriminator(nn.Module):
    def __init__(self, input_shape):
        super(Discriminator, self).__init__()

        channels, height, width = input_shape  ## input_shape: (3, 256, 256)

        # Output shape is (1, 16, 16)
        self.output_shape = (1, 16, 16)

        def depthwise_separable_conv(in_filters, out_filters, stride=1, padding=1):
            """Depthwise separable convolution block"""
            layers = [
                # Depthwise convolution
                nn.Conv2d(in_filters, in_filters, kernel_size=3, stride=stride, padding=padding, groups=in_filters, bias=False),
                nn.InstanceNorm2d(in_filters),
                nn.LeakyReLU(0.2, inplace=True),
                # Pointwise convolution
                nn.Conv2d(in_filters, out_filters, kernel_size=1, stride=1, padding=0, bias=False),
                nn.InstanceNorm2d(out_filters),
                nn.LeakyReLU(0.2, inplace=True),
            ]
            return layers

        self.model = nn.Sequential(
            # First layer: Regular convolution
            nn.Conv2d(channels, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Second layer: Depthwise separable convolution
            *depthwise_separable_conv(64, 128, stride=2, padding=1),

            # Third layer: Depthwise separable convolution
            *depthwise_separable_conv(128, 256, stride=2, padding=1),

            # Fourth layer: Depthwise separable convolution
            *depthwise_separable_conv(256, 512, stride=2, padding=1),

            # Fifth layer: Depthwise separable convolution (maintain spatial dimensions)
            *depthwise_separable_conv(512, 512, stride=1, padding=1),

            # Sixth layer: Depthwise separable convolution (maintain spatial dimensions)
            *depthwise_separable_conv(512, 512, stride=1, padding=1),

            # Final layer: Regular convolution, output channels = 1, output size = (1, 16, 16)
            nn.Conv2d(512, 1, kernel_size=16, stride=1, padding=0, bias=False),
            nn.Sigmoid()  # Use Sigmoid to map output to [0, 1] range
        )

    def forward(self, img):
        return self.model(img)

In [ ]:


#########################################################################
############################  utils.py  ###################################

# Buffer for previously generated samples
class ReplayBuffer:
    def __init__(self, max_size=50):
        assert max_size > 0, "Empty buffer or trying to create a black hole. Be careful."
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, data):  # Push an image into the buffer and then pop one out
        to_return = []  # Ensure randomness of data to improve discriminator's ability to distinguish real and fake images
        for element in data.data:
            element = torch.unsqueeze(element, 0)
            if len(self.data) < self.max_size:  # If the buffer is not full, keep adding images
                self.data.append(element)
                to_return.append(element)
            else:
                if random.uniform(0, 1) > 0.5:  # If the buffer is full, with 50% probability, take an image from the buffer or use the current input image
                    i = random.randint(0, self.max_size - 1)
                    to_return.append(self.data[i].clone())
                    self.data[i] = element
                else:
                    to_return.append(element)
        return Variable(torch.cat(to_return))


# Set the learning rate as the initial learning rate multiplied by the value of the given lr_lambda function
class LambdaLR:
    def __init__(self, n_epochs, offset, decay_start_epoch):  # (n_epochs = 50, offset = epoch, decay_start_epoch = 30)
        assert (
                           n_epochs - decay_start_epoch) > 0, "Decay must start before the training session ends!"  # Assertion: Ensure n_epochs > decay_start_epoch
        self.n_epochs = n_epochs
        self.offset = offset
        self.decay_start_epoch = decay_start_epoch

    def step(self, epoch):  # return 1 - max(0, epoch - 30) / (50 - 30)
        return 1.0 - max(0, epoch + self.offset - self.decay_start_epoch) / (self.n_epochs - self.decay_start_epoch)

In [ ]:


# Hyperparameter configuration
parser = argparse.ArgumentParser()
parser.add_argument("--epoch", type=int, default=0, help="epoch to start training from")
parser.add_argument("--n_epochs", type=int, default=1, help="number of epochs of training")
parser.add_argument("--dataset_name", type=str, default="dataset/horse2zebra",
                    help="name of the dataset")
parser.add_argument("--batch_size", type=int, default=1, help="size of the batches")
parser.add_argument("--lr", type=float, default=0.0003, help="adam: learning rate")
parser.add_argument("--b1", type=float, default=0.5, help="adam: decay of first order momentum of gradient")
parser.add_argument("--b2", type=float, default=0.999, help="adam: decay of first order momentum of gradient")
parser.add_argument("--decay_epoch", type=int, default=0, help="epoch from which to start lr decay")
parser.add_argument("--n_cpu", type=int, default=2, help="number of cpu threads to use during batch generation")
parser.add_argument("--img_height", type=int, default=256, help="size of image height")
parser.add_argument("--img_width", type=int, default=256, help="size of image width")
parser.add_argument("--channels", type=int, default=3, help="number of image channels")
parser.add_argument("--sample_interval", type=int, default=100, help="interval between saving generator outputs")
parser.add_argument("--checkpoint_interval", type=int, default=-1, help="interval between saving model checkpoints")
parser.add_argument("--n_residual_blocks", type=int, default=9, help="number of residual blocks in generator")
parser.add_argument("--lambda_cyc", type=float, default=10.0, help="cycle loss weight")
parser.add_argument("--lambda_id", type=float, default=5.0, help="identity loss weight")
opt = parser.parse_args()
# opt = parser.parse_args(args=[])                 ## Use this line when running in colab
print(opt)

# Create directories
os.makedirs("images/%s" % opt.dataset_name, exist_ok=True)
os.makedirs("save/%s" % opt.dataset_name, exist_ok=True)

# input_shape:(3, 256, 256)
input_shape = (opt.channels, opt.img_height, opt.img_width)

# Create generator and discriminator objects
G_AB = Generator(input_shape, opt.n_residual_blocks)
G_BA = Generator(input_shape, opt.n_residual_blocks)
D_A = Discriminator(input_shape)
D_B = Discriminator(input_shape)

# Loss functions
# MES: Binary cross-entropy
# L1loss: Preserves edges better compared to L2 Loss
criterion_GAN = torch.nn.MSELoss()
loss_fn_lpips = lpips.LPIPS(net='alex')
criterion_identity = torch.nn.L1Loss()

# If GPU is available, run in CUDA mode
if torch.cuda.is_available():
    G_AB = G_AB.cuda()
    G_BA = G_BA.cuda()
    D_A = D_A.cuda()
    D_B = D_B.cuda()
    criterion_GAN.cuda()
    loss_fn_lpips.cuda()
    criterion_identity.cuda()

# If epoch == 0, initialize model parameters; if epoch == n, load the pre-trained model trained up to the nth epoch
if opt.epoch != 0:
    # Load the pre-trained model trained up to the nth epoch
    G_AB.load_state_dict(torch.load("save/%s/G_AB_%d.pth" % (opt.dataset_name, opt.epoch)))
    G_BA.load_state_dict(torch.load("save/%s/G_BA_%d.pth" % (opt.dataset_name, opt.epoch)))
    D_A.load_state_dict(torch.load("save/%s/D_A_%d.pth" % (opt.dataset_name, opt.epoch)))
    D_B.load_state_dict(torch.load("save/%s/D_B_%d.pth" % (opt.dataset_name, opt.epoch)))
else:
    # Initialize model parameters
    G_AB.apply(weights_init_normal)
    G_BA.apply(weights_init_normal)
    D_A.apply(weights_init_normal)
    D_B.apply(weights_init_normal)

# Define optimization function, learning rate is 0.0003
optimizer_G = torch.optim.Adam(
    itertools.chain(G_AB.parameters(), G_BA.parameters()), lr=opt.lr, betas=(opt.b1, opt.b2)
)
optimizer_D_A = torch.optim.Adam(D_A.parameters(), lr=opt.lr, betas=(opt.b1, opt.b2))
optimizer_D_B = torch.optim.Adam(D_B.parameters(), lr=opt.lr, betas=(opt.b1, opt.b2))

# Learning rate update process
lr_scheduler_G = torch.optim.lr_scheduler.LambdaLR(
    optimizer_G, lr_lambda=LambdaLR(opt.n_epochs, opt.epoch, opt.decay_epoch).step
)
lr_scheduler_D_A = torch.optim.lr_scheduler.LambdaLR(
    optimizer_D_A, lr_lambda=LambdaLR(opt.n_epochs, opt.epoch, opt.decay_epoch).step
)
lr_scheduler_D_B = torch.optim.lr_scheduler.LambdaLR(
    optimizer_D_B, lr_lambda=LambdaLR(opt.n_epochs, opt.epoch, opt.decay_epoch).step
)

# Buffer for previously generated samples
fake_A_buffer = ReplayBuffer()
fake_B_buffer = ReplayBuffer()

# Image transformations
transforms_ = [
    transforms.Resize(int(opt.img_height * 1.12)),  # Resize image by 1.12 times
    transforms.RandomCrop((opt.img_height, opt.img_width)),  # Randomly crop to original size
    transforms.RandomHorizontalFlip(),  # Random horizontal flip
    transforms.ToTensor(),  # Convert to Tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # Normalize
]

# Training data loader
dataloader = DataLoader(  # Change to your own directory
    ImageDataset("dataset/horse2zebra", transforms_=transforms_, unaligned=True),
    # "./datasets/facades" , unaligned: Set unaligned data
    batch_size=opt.batch_size,  # batch_size = 1
    shuffle=True,
    num_workers=opt.n_cpu,
)
# Test data loader
val_dataloader = DataLoader(
    ImageDataset("dataset/horse2zebra", transforms_=transforms_, unaligned=True, mode="test"),  # "./datasets/facades"
    batch_size=5,
    shuffle=True,
    num_workers=1,
)


# Save generated samples from the test set every 100 iterations
def sample_images(batches_done):  # (100/200/300/400...)
    """Save generated samples from the test set"""
    imgs = next(iter(val_dataloader))  # Take one image
    G_AB.eval()
    G_BA.eval()
    real_A = Variable(imgs["A"]).cuda()  # Take a real A
    fake_B = G_AB(real_A)  # Generate fake B from real A
    real_B = Variable(imgs["B"]).cuda()  # Take a real B
    fake_A = G_BA(real_B)  # Generate fake A from real B
    # Arrange images along x-axis
    # make_grid(): Used to arrange several images in a grid
    real_A = make_grid(real_A, nrow=5, normalize=True)
    real_B = make_grid(real_B, nrow=5, normalize=True)
    fake_A = make_grid(fake_A, nrow=5, normalize=True)
    fake_B = make_grid(fake_B, nrow=5, normalize=True)
    # Arrange images along y-axis
    # Concatenate all images and save as one large image
    image_grid = torch.cat((real_A, fake_B, real_B, fake_A), 1)
    save_image(image_grid, "images/%s/%s.png" % (opt.dataset_name, batches_done), normalize=False)


def train():
    # ----------
    #  Training
    # ----------
    prev_time = time.time()  # Start time
    for epoch in range(opt.epoch, opt.n_epochs):  # for epoch in (0, 50)
        for i, batch in enumerate(
                dataloader):  # batch is a dict, batch['A']:(1, 3, 256, 256), batch['B']:(1, 3, 256, 256)
            #       print('here is %d' % i)
            # Read real images from the dataset
            # Convert tensor to Variable for computation graph, tensor must be converted to variable for backpropagation
            real_A = Variable(batch["A"]).cuda()  # Real image A
            real_B = Variable(batch["B"]).cuda()  # Real image B

            # Labels for real and fake images
            valid = Variable(torch.ones((real_A.size(0), *D_A.output_shape)),
                             requires_grad=False).cuda()  # Define real image label as 1 ones((1, 1, 16, 16))
            fake = Variable(torch.zeros((real_A.size(0), *D_A.output_shape)),
                            requires_grad=False).cuda()  # Define fake image label as 0 zeros((1, 1, 16, 16))

            ## -----------------
            ##  Train Generator
            ## Principle: The goal is to make the generated fake images be judged as real by the discriminator.
            ## During this process, the discriminator is fixed, and the fake images are passed to the discriminator.
            ## The result is compared with the real label.
            ## The parameters updated through backpropagation are those in the generator.
            ## This way, the generator is trained to produce images that the discriminator thinks are real, achieving the adversarial goal.
            ## -----------------
            G_AB.train()
            G_BA.train()

            ## Identity loss                                              ## A-style image placed in the B -> A generator should also produce A-style images
            loss_id_A = criterion_identity(G_BA(real_A),
                                           real_A)  ## loss_id_A is the loss when image A1 is passed through the B2A generator. The generated image A2 should also be in A-style, and the difference between A1 and A2 should be small.
            loss_id_B = criterion_identity(G_AB(real_B), real_B)

            loss_identity = (loss_id_A + loss_id_B) / 2  ## Identity loss

            ## GAN loss
            fake_B = G_AB(real_A)  ## Generate fake image B from real image A
            loss_GAN_AB = criterion_GAN(D_B(fake_B), valid)  ## Use discriminator B to judge fake image B. The goal of training the generator is to make the discriminator think the fake is real.
            fake_A = G_BA(real_B)  ## Generate fake image A from real image B
            loss_GAN_BA = criterion_GAN(D_A(fake_A), valid)  ## Use discriminator A to judge fake image A. The goal of training the generator is to make the discriminator think the fake is real.

            loss_GAN = (loss_GAN_AB + loss_GAN_BA) / 2  ## GAN loss

            # Cycle Consistency Loss with LPIPS
            recov_A = G_BA(fake_B)
            loss_cycle_A = loss_fn_lpips(recov_A, real_A).mean()  # Use LPIPS to calculate perceptual loss
            recov_B = G_AB(fake_A)
            loss_cycle_B = loss_fn_lpips(recov_B, real_B).mean()  # Use LPIPS to calculate perceptual loss

            loss_cycle = (loss_cycle_A + loss_cycle_B) / 2  # Take the average

            # Total loss                                                  ## Sum of all losses above
            loss_G = loss_GAN + opt.lambda_cyc * loss_cycle + opt.lambda_id * loss_identity
            optimizer_G.zero_grad()  ## Zero the gradients before backpropagation
            loss_G.backward()  ## Backpropagate the error
            optimizer_G.step()  ## Update parameters

            ## -----------------------
            ## Train Discriminator A
            ## Divided into two parts: 1. Judge real images as real; 2. Judge fake images as fake
            ## -----------------------
            ## Judge real images as real
            loss_real = criterion_GAN(D_A(real_A), valid)
            ## Judge fake images as fake (randomly take one from the previous buffer)
            fake_A_ = fake_A_buffer.push_and_pop(fake_A)
            loss_fake = criterion_GAN(D_A(fake_A_.detach()), fake)
            # Total loss
            loss_D_A = (loss_real + loss_fake) / 2
            optimizer_D_A.zero_grad()  ## Zero the gradients before backpropagation
            loss_D_A.backward()  ## Backpropagate the error
            optimizer_D_A.step()  ## Update parameters

            ## -----------------------
            ## Train Discriminator B
            ## Divided into two parts: 1. Judge real images as real; 2. Judge fake images as fake
            ## -----------------------
            # Judge real images as real
            loss_real = criterion_GAN(D_B(real_B), valid)
            ## Judge fake images as fake (randomly take one from the previous buffer)
            fake_B_ = fake_B_buffer.push_and_pop(fake_B)
            loss_fake = criterion_GAN(D_B(fake_B_.detach()), fake)
            # Total loss
            loss_D_B = (loss_real + loss_fake) / 2
            optimizer_D_B.zero_grad()  ## Zero the gradients before backpropagation
            loss_D_B.backward()  ## Backpropagate the error
            optimizer_D_B.step()  ## Update parameters
            loss_D = (loss_D_A + loss_D_B) / 2

            ## ----------------------
            ##  Log Progress
            ## ----------------------

            ## Determine the approximate remaining time. Assume current epoch = 5, i = 100
            batches_done = epoch * len(dataloader) + i  ## Time already trained: 5 * 400 + 100 iterations
            batches_left = opt.n_epochs * len(dataloader) - batches_done  ## Remaining iterations: 50 * 400 - 2100
            time_left = datetime.timedelta(
                seconds=batches_left * (time.time() - prev_time))  ## Time left = remaining iterations * time per iteration
            prev_time = time.time()
            # Print log
            sys.stdout.write(
                "\r[Epoch %d/%d] [Batch %d/%d] [D loss: %f] [G loss: %f, adv: %f, cycle: %f, identity: %f] ETA: %s"
                % (
                    epoch,
                    opt.n_epochs,
                    i,
                    len(dataloader),
                    loss_D.item(),
                    loss_G.item(),
                    loss_GAN.item(),
                    loss_cycle.item(),
                    loss_identity.item(),
                    time_left,
                )
            )
            # Save a set of images from the test set every 100 iterations
            if batches_done % opt.sample_interval == 0:
                sample_images(batches_done)

        # Update learning rate
        lr_scheduler_G.step()
        lr_scheduler_D_A.step()
        lr_scheduler_D_B.step()

    ## Save the model after training
    torch.save(G_AB.state_dict(), "save/%s/G_AB_%d.pth" % (opt.dataset_name, epoch))
    torch.save(G_BA.state_dict(), "save/%s/G_BA_%d.pth" % (opt.dataset_name, epoch))
    torch.save(D_A.state_dict(), "save/%s/D_A_%d.pth" % (opt.dataset_name, epoch))
    torch.save(D_B.state_dict(), "save/%s/D_B_%d.pth" % (opt.dataset_name, epoch))
    print("\nsave my model finished !!")
    #    ## Save the model every few epochs
    #     if opt.checkpoint_interval != -1 and epoch % opt.checkpoint_interval == 0:
    #         # Save model checkpoints
    #         torch.save(G_AB.state_dict(), "saved_models/%s/G_AB_%d.pth" % (opt.dataset_name, epoch))
    #         torch.save(G_BA.state_dict(), "saved_models/%s/G_BA_%d.pth" % (opt.dataset_name, epoch))
    #         torch.save(D_A.state_dict(), "saved_models/%s/D_A_%d.pth" % (opt.dataset_name, epoch))
    #         torch.save(D_B.state_dict(), "saved_models/%s/D_B_%d.pth" % (opt.dataset_name, epoch))


if __name__ == '__main__':
    train()  ## Train the model

In [ ]:
import os
import random
import torch
import datetime
import time
import sys
import argparse
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.autograd import Variable
from torchvision.utils import save_image, make_grid


def test():
    ## Hyperparameter settings
    parser = argparse.ArgumentParser()
    parser.add_argument('--batchSize', type=int, default=2, help='size of the batches')
    parser.add_argument('--dataroot', type=str, default='dataset/horse2zebra', help='root directory of the dataset')
    parser.add_argument('--channels', type=int, default=3, help='number of channels of input data')
    parser.add_argument('--n_residual_blocks', type=int, default=9, help='number of channels of output data')
    parser.add_argument('--size', type=int, default=256, help='size of the data (squared assumed)')
    parser.add_argument('--cuda', type=bool, default=True, help='use GPU computation')
    parser.add_argument('--n_cpu', type=int, default=8, help='number of cpu threads to use during batch generation')
    parser.add_argument('--generator_A2B', type=str, default='save/dataset/horse2zebra/G_AB_4.pth',
                        help='A2B generator checkpoint file')
    parser.add_argument('--generator_B2A', type=str, default='save/dataset/horse2zebra/G_BA_4.pth',
                        help='B2A generator checkpoint file')
    opt = parser.parse_args()
    print(opt)

    #################################
    ##         Test Preparation    ##
    #################################

    ## input_shape: (3, 256, 256)
    input_shape = (opt.channels, opt.size, opt.size)
    ## Create generator and discriminator objects
    netG_A2B = Generator(input_shape, opt.n_residual_blocks)
    netG_B2A = Generator(input_shape, opt.n_residual_blocks)

    ## Use CUDA
    if opt.cuda:
        netG_A2B.cuda()
        netG_B2A.cuda()

    checkpoint_1 = torch.load(opt.generator_A2B)
    netG_A2B.load_state_dict({k.replace('module.',''):v for k,v in checkpoint_1[opt.generator_A2B].items()})
    checkpoint_2 = torch.load(opt.generator_B2A)
    netG_A2B.load_state_dict({k.replace('module.',''):v for k,v in checkpoint_2[opt.generator_B2A].items()})

    #netG_A2B.load_state_dict(torch.load(opt.generator_A2B), strict=False)
    #netG_B2A.load_state_dict(torch.load(opt.generator_B2A), strict=False)

    ## Set to evaluation mode
    netG_A2B.eval()
    netG_B2A.eval()

    ## Create a tensor array
    Tensor = torch.cuda.FloatTensor if opt.cuda else torch.Tensor
    input_A = Tensor(opt.batchSize, opt.channels, opt.size, opt.size)
    input_B = Tensor(opt.batchSize, opt.channels, opt.size, opt.size)

    '''Build the test dataset'''
    transforms_ = [transforms.ToTensor(),
                   transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
    dataloader = DataLoader(ImageDataset(opt.dataroot, transforms_=transforms_, mode='test'),
                            batch_size=opt.batchSize, shuffle=False, num_workers=opt.n_cpu)

    #################################
    ##           Start Test        ##
    #################################

    '''If the file path does not exist, create one (to store test output images)'''
    if not os.path.exists('output/A'):
        os.makedirs('output/A')
    if not os.path.exists('output/B'):
        os.makedirs('output/B')

    for i, batch in enumerate(dataloader):
        ## Input data (real)
        real_A = Variable(input_A.copy_(batch['A']))
        real_B = Variable(input_B.copy_(batch['B']))
        ## Generated data (fake) through the generator
        fake_B = 0.5 * (netG_A2B(real_A).data + 1.0)
        fake_A = 0.5 * (netG_B2A(real_B).data + 1.0)
        ## Save images
        save_image(fake_A, 'output/A/%04d.png' % (i + 1))
        save_image(fake_B, 'output/B/%04d.png' % (i + 1))
        print('processing (%04d)-th image...' % (i))
    print("Testing completed")


if __name__ == '__main__':
    test()  ## Test the model